In [ ]:
"""
coco_to_tsv.py
==============
Convertit un fichier d'annotation COCO (avec track_id) en fichier TSV
structuré pour analyses statistiques et biologiques (tracking d'Aplysia).

Colonnes produites :
  track_id | label | image_id | frame_index | file_name
  bbox_x | bbox_y | bbox_w | bbox_h | bbox_cx | bbox_cy
  area | seg_n_points | seg_n_polygons
  bbox_aspect_ratio | bbox_diag | bbox_fill_ratio

Usage :
  python coco_to_tsv.py --input Annotation_COCO.json --output annotations.tsv

Options :
  --input     Chemin vers le fichier JSON COCO (défaut : Annotation_COCO.json)
  --output    Chemin vers le fichier TSV de sortie (défaut : annotations.tsv)
  --no-seg    Ne pas inclure les colonnes de segmentation (plus léger)
"""

import json
import csv
import math
import argparse
from pathlib import Path


# ---------------------------------------------------------------------------
# Helpers géométriques
# ---------------------------------------------------------------------------

def bbox_diagonal(w, h):
    """Diagonale de la bounding box."""
    return round(math.sqrt(w ** 2 + h ** 2), 4)

def bbox_fill_ratio(area, w, h):
    """Ratio aire réelle / aire bbox — proche de 1 = objet rectangulaire."""
    bbox_area = w * h
    if bbox_area == 0:
        return None
    return round(area / bbox_area, 6)

def seg_perimeter(segmentation):
    """Périmètre total du polygone de segmentation (en pixels)."""
    total = 0.0
    for poly in segmentation:
        # Convertir la liste plate [x0,y0,x1,y1,...] en liste de points
        points = [(poly[i], poly[i+1]) for i in range(0, len(poly) - 1, 2)]
        for j in range(len(points)):
            x1, y1 = points[j]
            x2, y2 = points[(j + 1) % len(points)]  # boucle sur le dernier point
            total += math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
    return round(total, 4)

# ---------------------------------------------------------------------------
# Conversion principale
# ---------------------------------------------------------------------------

def coco_to_tsv(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # -- Index des images : image_id -> {frame_index, file_name, width, height}
    image_index = {
        img["id"]: img
        for img in coco.get("images", [])
    }

    # -- Index des catégories : category_id -> nom
    cat_index = {
        cat["id"]: cat["name"]
        for cat in coco.get("categories", [])
    }

    # -- Colonnes de base
    fieldnames = [
        "image_id",
        "frame_index",
        "file_name",
        "track_id",
        "label",
        # Bounding box
        "bbox_x",       # coin supérieur gauche X
        "bbox_y",       # coin supérieur gauche Y
        "bbox_w",       # largeur
        "bbox_h",       # hauteur
        "bbox_cx",      # centroïde X
        "bbox_cy",      # centroïde Y
        # Aire et forme
        "area",
        "bbox_diag",           # diagonale (px)
        "bbox_fill_ratio",     # area / (w*h)
        "seg_perimeter",
        "circularity",
    ]

    # -- Écriture du TSV
    rows_written = 0
    with open(output_path, "w", newline="", encoding="utf-8") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()

        for ann in coco.get("annotations", []):
            image_id  = ann["image_id"]
            img_info  = image_index.get(image_id, {})
            label     = cat_index.get(ann.get("category_id"), "unknown")

            bbox = ann.get("bbox", [None, None, None, None])
            bx, by, bw, bh = bbox
            
            # Centroïde de la bbox
            cx = round(bx + bw / 2, 4) if bw is not None else None
            cy = round(by + bh / 2, 4) if bh is not None else None

            area = ann.get("area")
            seg  = ann.get("segmentation", [])

            row = {
                "image_id":          image_id,
                "frame_index":       img_info.get("frame_index"),
                "file_name":         img_info.get("file_name"),
                "track_id":          ann.get("track_id"),
                "label":             label,
                "bbox_x":            bx,
                "bbox_y":            by,
                "bbox_w":            bw,
                "bbox_h":            bh,
                "bbox_cx":           cx,
                "bbox_cy":           cy,
                "bbox_diag":         bbox_diagonal(bw, bh) if bw and bh else None,
                "area":              area,
                "bbox_fill_ratio":   bbox_fill_ratio(area, bw, bh)
                                     if area and bw and bh else None,
                "seg_perimeter": seg_perimeter(seg),
                "circularity": round((4 * math.pi * area) / (seg_perimeter(seg) ** 2), 6) # = 1.0 pour un cercle parfait, < 1 pour une forme allongée
            }

            writer.writerow(row)
            rows_written += 1

    print(f"[OK] {rows_written} annotations écrites dans : {output_path}")
    print(f"     Colonnes : {len(fieldnames)}")
    print(f"     Séparateur : tabulation (TSV)")

def load_galaxy_inputs(path: str = "galaxy_inputs/galaxy_inputs.json") -> dict:
    p = Path(path)
    with open(p, encoding="utf-8") as f:
        return json.load(f)
# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def main():
    galaxy = load_galaxy_inputs("galaxy_inputs/galaxy_inputs.json")
    input_path    = galaxy["Annotation"][0]["path"]
    output_path = "outputs/annotations.tsv"
    coco_to_tsv(
        input_path=str(input_path),
        output_path=str(output_path)
    )

if __name__ == "__main__":
    main()
